# 01 | Olist Dataset Audit

**Purpose:** Inspect the nine Olist source tables, understand their structure, and establish a first-pass data-quality baseline.

**Inputs:** The nine original Olist CSV files in `data/raw/` (downloaded separately; raw data is not stored in this repository).

**Outputs:** Dataset-level row/column counts, missing-value summaries, duplicate-row counts, and CSV audit reports saved under `data/processed/`.

Run this notebook first. Then proceed to the identifier and relationship analysis notebooks.


In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Environment ready!")

In [ ]:
# Set this environment variable when your data is stored elsewhere:
# OLIST_DATA_DIR=/path/to/olist/data/raw
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = Path(os.environ.get("OLIST_DATA_DIR", PROJECT_ROOT / "data" / "raw")).expanduser()

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Olist CSV folder not found: {DATA_DIR}\n"
        "Download the Olist CSV files and place them in data/raw, "
        "or set the OLIST_DATA_DIR environment variable."
    )

print("Dataset folder:", DATA_DIR)


## Load and inspect the source tables


In [ ]:
required_csv_files = [
    "olist_orders_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv",
]

available_csvs = {file.name: file for file in DATA_DIR.glob("*.csv")}
missing_files = sorted(set(required_csv_files) - set(available_csvs))

if missing_files:
    raise FileNotFoundError(
        "The following required Olist CSV files are missing: "
        + ", ".join(missing_files)
    )

csv_files = [available_csvs[name] for name in required_csv_files]
extra_csvs = sorted(set(available_csvs) - set(required_csv_files))

print("Required source CSV files found:", len(csv_files))
if extra_csvs:
    print("Additional CSVs present (not included in this source audit):", extra_csvs)
for file in csv_files:
    print(file.name)

In [ ]:
datasets = {}

for file in csv_files:

    dataset_name = file.stem

    df = pd.read_csv(file, low_memory=False)

    datasets[dataset_name] = df

    print("\n" + "="*60)
    print("DATASET:", dataset_name)
    print("="*60)

    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nFirst 3 records:")
    display(df.head(3))

## Summarize data quality


In [ ]:
audit_results = []

for name, df in datasets.items():

    audit_results.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum(),
        "Memory (MB)": round(
            df.memory_usage(deep=True).sum() / (1024**2), 2
        )
    })

audit_df = pd.DataFrame(audit_results)

display(audit_df)

In [ ]:
missing_reports = []

for name, df in datasets.items():

    missing = df.isnull().sum()

    for column, count in missing.items():

        if count > 0:

            missing_reports.append({
                "Dataset": name,
                "Column": column,
                "Missing Count": count,
                "Missing Percentage": round(
                    (count / len(df)) * 100, 2
                )
            })

missing_df = pd.DataFrame(missing_reports)

display(missing_df)

## Export audit summaries


In [ ]:
OUTPUT_DIR = DATA_DIR.parent / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

audit_df.to_csv(OUTPUT_DIR / "dataset_audit_summary.csv", index=False)
missing_df.to_csv(OUTPUT_DIR / "missing_values_report.csv", index=False)

print("Audit reports saved to:", OUTPUT_DIR)
